# Simple, fast baseline: Logistic Regression on one-hot / scaled features.

Purpose: establish the "floor" score before trying gradient boosting.
Everything downstream (03/04/05) should beat this comfortably.


In [6]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

PROJECT_DIR = os.getcwd()  
sys.path.insert(0, PROJECT_DIR)

fe = importlib.import_module("06_Feature_Engineering")
cv = importlib.import_module("07_Cross_Validation")

MODEL_NAME = "baseline_logreg"
ARTIFACT_DIR = "./artifacts"
SUB_PATH = f"{ARTIFACT_DIR}/test_pred_{MODEL_NAME}.csv"
OOF_PATH = f"{ARTIFACT_DIR}/oof_{MODEL_NAME}.npy"

In [7]:
def build_pipeline(num_cols, cat_cols):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ]
    )
    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",  # target is imbalanced (~17.5% positive)
        random_state=cv.SEED,
    )
    return Pipeline([("prep", preprocessor), ("clf", model)])


In [8]:
def main():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)

    train, test = fe.load_raw_data()
    train = fe.engineer_features(train)
    test = fe.engineer_features(test)
    num_cols, cat_cols = fe.get_feature_lists(train)

    y = train[fe.TARGET].values
    fold_ids = cv.get_or_create_folds(train, target_col=fe.TARGET)

    oof_pred = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    fold_scores = []

    print("=" * 70)
    print(f"BASELINE: Logistic Regression ({cv.N_SPLITS}-fold CV)")
    print("=" * 70)

    for fold in range(cv.N_SPLITS):
        train_idx, valid_idx = cv.fold_split(train, fold_ids, fold)

        X_train = train.loc[train_idx, num_cols + cat_cols]
        X_valid = train.loc[valid_idx, num_cols + cat_cols]
        y_train, y_valid = y[train_idx], y[valid_idx]

        pipe = build_pipeline(num_cols, cat_cols)
        pipe.fit(X_train, y_train)

        valid_pred = pipe.predict_proba(X_valid)[:, 1]
        oof_pred[valid_idx] = valid_pred

        fold_auc = roc_auc_score(y_valid, valid_pred)
        fold_scores.append(fold_auc)
        print(f"Fold {fold}: AUC = {fold_auc:.5f}")

        test_pred += pipe.predict_proba(test[num_cols + cat_cols])[:, 1] / cv.N_SPLITS

    print(f"\nMean fold AUC: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
    cv.summarize_oof(y, oof_pred, MODEL_NAME)

    np.save(OOF_PATH, oof_pred)
    pd.DataFrame({fe.ID_COL: test[fe.ID_COL], fe.TARGET: test_pred}).to_csv(
        SUB_PATH, index=False
    )
    print(f"\nSaved OOF predictions -> {OOF_PATH}")
    print(f"Saved test predictions -> {SUB_PATH}")

In [9]:
if __name__ == "__main__":
    main()

BASELINE: Logistic Regression (5-fold CV)
Fold 0: AUC = 0.93681
Fold 1: AUC = 0.93805
Fold 2: AUC = 0.93924
Fold 3: AUC = 0.93875
Fold 4: AUC = 0.93815

Mean fold AUC: 0.93820 (+/- 0.00082)
[baseline_logreg] OOF ROC-AUC: 0.93820

Saved OOF predictions -> ./artifacts/oof_baseline_logreg.npy
Saved test predictions -> ./artifacts/test_pred_baseline_logreg.csv
